In [ ]:
import pandas as pd

Download the AGH weather measurments data:

In [2]:
df = pd.read_csv("Data/combined.csv")

In [3]:
# set the time column to datetime, sort the values by it and set is as index
df["time"] = pd.to_datetime(df["time"])
df = df.sort_values(by="time")
df = df.set_index("time")
df.index = df.index.floor("h")

In [4]:
df.head()

,averageAirTemp,averageWindchill,averageHeatindex,averageDewPointTemperature,averageRelativeHumidity,averageAirPressure,averageSeaLevelPressure,averageWindDirection,maxWindSpeed,averageWindSpeed,rainAccumulation,rainIntensity,averagePm10
time,,,,,,,,,,,,,
2016-01-01 00:00:00,-8.871667,-8.296667,129.668333,-13.201667,70.875000,1000.903333,1027.875000,280.0,1.7,0.771667,0.0,0.0,NaN
2016-01-01 01:00:00,-9.153333,-8.486667,132.121667,-13.351667,71.593333,1000.516667,1027.516667,265.0,1.1,0.703333,0.0,0.0,NaN
2016-01-01 02:00:00,-9.475000,-8.795000,134.566667,-13.636667,71.685000,1000.363333,1027.363333,269.0,1.4,0.698333,0.0,0.0,NaN
2016-01-01 03:00:00,-9.851667,-9.246667,137.543333,-13.971667,71.863333,1000.123333,1027.180000,283.0,1.6,0.743333,0.0,0.0,NaN
2016-01-01 04:00:00,-10.175000,-9.460000,139.920000,-14.321667,71.668333,1000.190000,1027.290000,286.0,1.4,0.661667,0.0,0.0,NaN


### Check the quality of the AGH measurments data

In [5]:
# check if the index is unique
df.index[df.index.duplicated()]

DatetimeIndex(['2018-07-10 22:00:00', '2018-07-10 23:00:00',
               '2018-07-11 00:00:00', '2018-07-11 01:00:00',
               '2018-07-11 02:00:00', '2018-07-11 03:00:00',
               '2018-07-11 04:00:00', '2018-07-11 05:00:00',
               '2018-07-11 06:00:00', '2018-07-11 07:00:00',
               '2018-07-11 08:00:00', '2018-07-11 09:00:00',
               '2018-07-11 10:00:00', '2018-07-11 11:00:00',
               '2018-07-11 12:00:00', '2018-07-11 13:00:00',
               '2018-07-11 14:00:00', '2018-07-11 15:00:00',
               '2018-07-11 16:00:00', '2018-07-11 17:00:00',
               '2018-07-11 18:00:00'],
              dtype='datetime64[ns]', name='time', freq=None)

In [6]:
# remove duplicated rows
df = df[~df.index.duplicated(keep="first")]

In [7]:
# there should not be duplicates anymore
df.index[df.index.duplicated()]

DatetimeIndex([], dtype='datetime64[ns]', name='time', freq=None)

Check if the `time` includes all possible hours within the time period of intrest

In [8]:
full_time_range = pd.date_range(
    start="2016-01-01 00:00:00", end="2025-09-30 23:00:00", freq="h"
)
missing_agh_hours = full_time_range.difference(df.index)
print("The size of missing AGH hours: ", missing_agh_hours.size)

print("Is the index unique: ", df.index.is_unique)

The size of missing AGH hours:  1886
Is the index unique:  True


Additionally there are some empty data points, which is not good for timeseries:

In [9]:
df.isna().mean() * 100

averageAirTemp                 2.437489
averageWindchill               2.437489
averageHeatindex               2.437489
averageDewPointTemperature     2.437489
averageRelativeHumidity        2.437489
averageAirPressure             2.437489
averageSeaLevelPressure        2.437489
averageWindDirection           1.464125
maxWindSpeed                   1.464125
averageWindSpeed               1.464125
rainAccumulation               0.357871
rainIntensity                  0.357871
averagePm10                   25.869324
dtype: float64

---
#### To fill the gaps in the AGH measurments data (in `df`) I will use the open meteo dataset:
(open meteo dataset)[https://open-meteo.com/en/docs/historical-weather-api?hourly=temperature_2m,dew_point_2m,relative_humidity_2m,surface_pressure,pressure_msl,wind_direction_10m,wind_speed_10m,cloud_cover_low&start_date=2016-01-01&end_date=2025-12-31&latitude=50.06143&longitude=19.93658&wind_speed_unit=ms&timezone=Europe%2FBerlin]


In [10]:
# open meteo dataframe
om_df = pd.read_csv(filepath_or_buffer="./Data/open-meteo.csv", skiprows=3)
om_df["time"] = pd.to_datetime(om_df["time"])
om_df = om_df.sort_values(by="time")
om_df = om_df.set_index("time")
om_df.index = om_df.index.floor("h")

In [11]:
om_df.head()

,averageAirTemp,averageDewPointTemperature,averageRelativeHumidity,averageAirPressure,averageSeaLevelPressure,averageWindDirection,averageWindSpeed,cloud_cover_low (%)
time,,,,,,,,
2016-01-01 00:00:00,-6.7,-11.3,70,1001.9,1029.7,360,0.10,0
2016-01-01 01:00:00,-7.4,-11.4,73,1001.8,1029.7,304,0.36,0
2016-01-01 02:00:00,-7.9,-11.4,76,1001.4,1029.3,277,0.81,0
2016-01-01 03:00:00,-7.6,-11.5,74,1001.1,1029.0,270,0.80,0
2016-01-01 04:00:00,-7.3,-11.6,72,1000.9,1028.7,270,0.40,0


#### Check the quality of open-meteo dataset

In [12]:
# check if the index is unique
om_df.index[om_df.index.duplicated()]

DatetimeIndex([], dtype='datetime64[ns]', name='time', freq=None)

In [13]:
missing_om_hours = full_time_range.difference(om_df.index)
print("The size of missing open-meteo hours: ", missing_om_hours.size)

print("Is index unique: ", om_df.index.is_unique)

The size of missing open-meteo hours:  0
Is index unique:  True


In [14]:
# check if there are empty data-points:
om_df.isna().mean() * 100

averageAirTemp                0.0
averageDewPointTemperature    0.0
averageRelativeHumidity       0.0
averageAirPressure            0.0
averageSeaLevelPressure       0.0
averageWindDirection          0.0
averageWindSpeed              0.0
cloud_cover_low (%)           0.0
dtype: float64

---
#### The open-meteo data set is clean, so it can be used to clean-up the AGH measurments
To do it:
1. Keep the open-meteo (`om_df`) datetime column in the `result_df`
2. Fill with the datapoints from AGH measurments (`df`) 
3. Fill empty data points with the data provided by open-meteo (`om_df`)

---

In [15]:
# 1.
result_df = pd.DataFrame()
result_df["time"] = om_df.index.copy()
result_df = result_df.set_index("time")
print("before")
print("shape of result: ", result_df.shape)
print("index in result_df: ", result_df.index.name)
print("columns in result_df: ", result_df.columns)

# 2.
result_df = result_df.merge(df, how="left", left_on="time", right_index=True)
result_df = result_df.drop(columns="averagePm10")
print("\nafter")
print("shape of result: ", result_df.shape)
print("index in result_df: ", result_df.index.name)
print("columns in result_df: ", result_df.columns)

before
shape of result:  (87672, 0)
index in result_df:  time
columns in result_df:  Index([], dtype='object')

after
shape of result:  (87672, 12)
index in result_df:  time
columns in result_df:  Index(['averageAirTemp', 'averageWindchill', 'averageHeatindex',
       'averageDewPointTemperature', 'averageRelativeHumidity',
       'averageAirPressure', 'averageSeaLevelPressure', 'averageWindDirection',
       'maxWindSpeed', 'averageWindSpeed', 'rainAccumulation',
       'rainIntensity'],
      dtype='object')


In [16]:
# quick check if the data is fine
print(result_df.index.is_unique)
missing_result_hours = full_time_range.difference(result_df.index)
print("The size of missing open-meteo hours: ", missing_result_hours.size)

True
The size of missing open-meteo hours:  0


#### Check if the amount of filled data is the same as before

In [17]:
# before
print(
    (df.shape[0] - df.isna().sum()).values[:-1]
    == (result_df.shape[0] - result_df.isna().sum()).values
)
# after
print(result_df.shape[0] - result_df.isna().sum())

[ True  True  True  True  True  True  True  True  True  True  True  True]
averageAirTemp                83694
averageWindchill              83694
averageHeatindex              83694
averageDewPointTemperature    83694
averageRelativeHumidity       83694
averageAirPressure            83694
averageSeaLevelPressure       83694
averageWindDirection          84529
maxWindSpeed                  84529
averageWindSpeed              84529
rainAccumulation              85478
rainIntensity                 85478
dtype: int64


In [18]:
# 3. Insert the data provided by open-meteo if NaN in result_df

# add the cloud cover to the dataset
result_df["cloud_cover_low (%)"] = None
result_df = result_df[list(om_df.columns)]
result_df = result_df.combine_first(om_df)

In [19]:
result_df.isna().mean() * 100

averageAirTemp                0.0
averageDewPointTemperature    0.0
averageRelativeHumidity       0.0
averageAirPressure            0.0
averageSeaLevelPressure       0.0
averageWindDirection          0.0
averageWindSpeed              0.0
cloud_cover_low (%)           0.0
dtype: float64

#### Data issue is resolved :) 
---

### Appending the PM10 data 
As the PM10 data I will use the one collected from the 3 PM10 measurments stations operated by GIOGS in krakow.
The dataset contains the data till the october of 2025.

In [20]:
# collect the GIOGS data
giogs_pm10_df = pd.read_csv("./Data/other_data_set.csv")
giogs_pm10_df["Time"] = pd.to_datetime(giogs_pm10_df["Time"])
giogs_pm10_df = giogs_pm10_df.sort_values("Time")
giogs_pm10_df = giogs_pm10_df.set_index("Time")
giogs_pm10_df = giogs_pm10_df["mean"]

In [21]:
giogs_pm10_df

Time
2016-01-01 00:00:00    313.267333
2016-01-01 01:00:00    288.007000
2016-01-01 02:00:00    298.940000
2016-01-01 03:00:00    289.741667
2016-01-01 04:00:00    296.505000
                          ...    
2025-09-30 20:00:00     12.666667
2025-09-30 21:00:00     15.033333
2025-09-30 22:00:00     15.633333
2025-09-30 23:00:00     16.700000
2025-10-01 00:00:00     14.200000
Name: mean, Length: 85464, dtype: float64

In [22]:
missing = full_time_range.difference(giogs_pm10_df.index)
print("missing hours: ")
print(missing)
print("Giogs index unique: ", giogs_pm10_df.index.is_unique)

missing hours: 
DatetimeIndex(['2025-03-30 02:00:00'], dtype='datetime64[ns]', freq='h')
Giogs index unique:  True


In [23]:
result_df[result_df.index == "2025-03-30 02:00:00"]

,averageAirTemp,averageDewPointTemperature,averageRelativeHumidity,averageAirPressure,averageSeaLevelPressure,averageWindDirection,averageWindSpeed,cloud_cover_low (%)
time,,,,,,,,
2025-03-30 02:00:00,7.456667,0.05,59.445,983.146667,1008.086667,326.0,0.583333,0


In [24]:
giogs_pm10_df.iloc[-1:]

Time
2025-10-01    14.2
Name: mean, dtype: float64

Here is the issue with different timezones.
The missing date - `2025-03-30 02:00:00` is missing because the clocks in Poland change into 
the summer time.

In [25]:
giogs_pm10_df[
    "2025-03-30 02:00:00"
] = -10  # setting negative number, because PM10 cant be negative!

In [26]:
print(giogs_pm10_df["2025-03-30 01:00:00"])
print(giogs_pm10_df["2025-03-30 02:00:00"])
print(giogs_pm10_df["2025-03-30 03:00:00"])
print(giogs_pm10_df["2025-03-30 04:00:00"])
print(giogs_pm10_df["2025-03-30 05:00:00"])

48.20000000000001
-10.0
47.46666666666667
39.66666666666666
40.36666666666667


In [27]:
giogs_pm10_df["2025-03-30 02:00:00"] = giogs_pm10_df["2025-03-30 03:00:00"]

In [28]:
print(giogs_pm10_df["2025-03-30 01:00:00"])
print(giogs_pm10_df["2025-03-30 02:00:00"])
print(giogs_pm10_df["2025-03-30 03:00:00"])
print(giogs_pm10_df["2025-03-30 04:00:00"])
print(giogs_pm10_df["2025-03-30 05:00:00"])

48.20000000000001
47.46666666666667
47.46666666666667
39.66666666666666
40.36666666666667


In [29]:
giogs_pm10_df.loc["2025-03-30 03:00:00":] = giogs_pm10_df.loc[
    "2025-03-30 03:00:00":
].shift(-1)

In [30]:
print(giogs_pm10_df["2025-03-30 01:00:00"])
print(giogs_pm10_df["2025-03-30 02:00:00"])
print(giogs_pm10_df["2025-03-30 03:00:00"])
print(giogs_pm10_df["2025-03-30 04:00:00"])
print(giogs_pm10_df["2025-03-30 05:00:00"])

48.20000000000001
47.46666666666667
39.66666666666666
40.36666666666667
40.23333333333333


In [31]:
giogs_pm10_df["2025-03-30 01:00:00":]

Time
2025-03-30 01:00:00    48.200000
2025-03-30 03:00:00    39.666667
2025-03-30 04:00:00    40.366667
2025-03-30 05:00:00    40.233333
2025-03-30 06:00:00    42.500000
                         ...    
2025-09-30 21:00:00    15.633333
2025-09-30 22:00:00    16.700000
2025-09-30 23:00:00    14.200000
2025-10-01 00:00:00          NaN
2025-03-30 02:00:00    47.466667
Name: mean, Length: 4440, dtype: float64

Now, since I plan to take only the dates till the middle of 2025, I dont care about 
the issue with the data at the end, the rest of the values are properly aligned   
and can be joined with the `result_df` dataframe.

In [32]:
giogs_pm10_df.index.is_unique

True

In [33]:
result_df = result_df.join(giogs_pm10_df, how="left")

In [34]:
# cut off the values till the middle of 2025:
result_df = result_df.loc[:"2025-07-01 00:00:00"]

In [35]:
result_df

,averageAirTemp,averageDewPointTemperature,averageRelativeHumidity,averageAirPressure,averageSeaLevelPressure,averageWindDirection,averageWindSpeed,cloud_cover_low (%),mean
time,,,,,,,,,
2016-01-01 00:00:00,-8.871667,-13.201667,70.875000,1000.903333,1027.875000,280.0,0.771667,0,313.267333
2016-01-01 01:00:00,-9.153333,-13.351667,71.593333,1000.516667,1027.516667,265.0,0.703333,0,288.007000
2016-01-01 02:00:00,-9.475000,-13.636667,71.685000,1000.363333,1027.363333,269.0,0.698333,0,298.940000
2016-01-01 03:00:00,-9.851667,-13.971667,71.863333,1000.123333,1027.180000,283.0,0.743333,0,289.741667
2016-01-01 04:00:00,-10.175000,-14.321667,71.668333,1000.190000,1027.290000,286.0,0.661667,0,296.505000
...,...,...,...,...,...,...,...,...,...
2025-06-30 20:00:00,19.476667,7.181667,44.883333,992.435000,1016.558333,343.0,1.366667,3,21.133333
2025-06-30 21:00:00,18.278333,7.105000,48.145000,992.688333,1016.916667,335.0,1.125000,3,22.200000
2025-06-30 22:00:00,17.361667,6.900000,50.253333,993.128333,1017.448333,305.0,0.750000,11,22.833333


The `'mean'` column referes to the mean PM10 values taken from 3 measurment stations in Kraków

---
### Quality of the final data set 

Amount of NaN values in the dataframe:

In [36]:
result_df.isna().sum()

averageAirTemp                0
averageDewPointTemperature    0
averageRelativeHumidity       0
averageAirPressure            0
averageSeaLevelPressure       0
averageWindDirection          0
averageWindSpeed              0
cloud_cover_low (%)           0
mean                          0
dtype: int64

Is the index (timedate) unique? 

In [37]:
result_df.index.is_unique

True

And the last sanity check of the data itself:
Check if the data makes sense:
 1. Are the values ok (cloud cover between 0-100, mean > 0 (PM10 concetration) and so on?

In [38]:
result_df.describe()

,averageAirTemp,averageDewPointTemperature,averageRelativeHumidity,averageAirPressure,averageSeaLevelPressure,averageWindDirection,averageWindSpeed,mean
count,83257.000000,83257.000000,83257.000000,83257.000000,83257.000000,83257.000000,83257.000000,83257.000000
mean,10.419419,2.945669,62.661409,989.637063,1014.540911,211.407389,1.766609,37.063318
std,8.691956,7.197963,16.151584,8.319349,8.695757,97.542690,1.159314,30.406464
min,-20.728333,-25.405000,6.410000,718.418182,736.363636,0.000000,0.100000,1.365107
25%,3.525000,-2.415000,51.468333,984.660000,1009.186667,105.000000,0.831667,18.433333
50%,9.788333,2.810000,65.858333,989.901667,1014.496667,251.000000,1.531667,28.352133
75%,17.133333,8.733333,75.740000,994.855000,1019.938333,281.000000,2.386667,44.930967
max,35.673333,20.700000,100.000000,1017.685000,1044.400000,360.000000,11.520000,402.945000


After carefully inspecting all values I conclude that the data set is ready!

In [39]:
result_df.to_csv("./Data/training_dataset.csv", index=True)
"""
df = pd.read_csv(
    './Data/training_dataset.csv',
    index_col=0,
    parse_dates=True
)
"""

"\ndf = pd.read_csv(\n    './Data/training_dataset.csv',\n    index_col=0,\n    parse_dates=True\n)\n"